In [6]:
import pandas as pd

Data cleaning and validation

objective -
clean the important data quality issues found during profiling and prepare reliable datasets for joining and analysis.

In [7]:
customers = pd.read_csv("../01_raw_data/olist_customers_dataset.csv")
orders = pd.read_csv("../01_raw_data/olist_orders_dataset.csv")
order_items = pd.read_csv("../01_raw_data/olist_order_items_dataset.csv")
products = pd.read_csv("../01_raw_data/olist_products_dataset.csv")
payments = pd.read_csv("../01_raw_data/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../01_raw_data/olist_order_reviews_dataset.csv")
sellers = pd.read_csv("../01_raw_data/olist_sellers_dataset.csv")
geolocation = pd.read_csv("../01_raw_data/olist_geolocation_dataset.csv")
category_translation = pd.read_csv(
    "../01_raw_data/product_category_name_translation.csv"
)

In [8]:
print("All datasets loaded successfully.")

All datasets loaded successfully.


In [9]:
customers_clean = customers.copy()

In [10]:
customers_clean["customer_zip_code_prefix"] = (
    customers_clean["customer_zip_code_prefix"]
    .astype("string")
    .str.zfill(5)
)

In [11]:
customers_clean["customer_zip_code_prefix"].str.len().value_counts()

customer_zip_code_prefix
5    99441
Name: count, dtype: Int64

In [12]:
sellers_clean = sellers.copy()
sellers_clean["seller_zip_code_prefix"] = (
    sellers_clean["seller_zip_code_prefix"]
    .astype("string")
    .str.zfill(5)
)

In [13]:
sellers_clean["seller_zip_code_prefix"].str.len().value_counts()

seller_zip_code_prefix
5    3095
Name: count, dtype: Int64

In [14]:
geolocation_clean = geolocation.copy()

In [15]:
geolocation_clean["geolocation_zip_code_prefix"] = (
    geolocation_clean["geolocation_zip_code_prefix"]
    .astype("string")
    .str.zfill(5)
)

In [16]:
geolocation_clean["geolocation_zip_code_prefix"].str.len().value_counts()

geolocation_zip_code_prefix
5    1000163
Name: count, dtype: Int64

In [17]:
geolocation_clean = geolocation_clean.drop_duplicates().copy()

In [18]:
print("rows after removing exact duplicates",geolocation_clean.shape[0])
print("exact duplicates remaining",geolocation_clean.duplicated().sum())

rows after removing exact duplicates 738332
exact duplicates remaining 0


In [19]:
postcode_state_conflicts = (
    geolocation_clean
    .groupby("geolocation_zip_code_prefix")["geolocation_state"]
    .nunique()
)
postcode_state_conflicts = postcode_state_conflicts[
    postcode_state_conflicts > 1
]

postcode_state_conflicts

geolocation_zip_code_prefix
02116    2
04011    2
21550    2
23056    2
72915    2
78557    2
79750    2
80630    2
Name: geolocation_state, dtype: int64

8 postcode state conflicts confirmed and flagged for handling during geolocation aggregation

In [20]:
conflict_postcodes = postcode_state_conflicts.index
(
    geolocation_clean[
        geolocation_clean["geolocation_zip_code_prefix"].isin(conflict_postcodes)
    ]
    .groupby(
        ["geolocation_zip_code_prefix","geolocation_state"]
    )
    .size()
    .reset_index(name = "row_count")
    .sort_values(
        ["geolocation_zip_code_prefix","row_count"],
        ascending=[True,False]
    )
)

,geolocation_zip_code_prefix,geolocation_state,row_count
1,02116,SP,10
0,02116,RN,1
3,04011,SP,69
2,04011,AC,1
5,21550,RJ,144
4,21550,AC,1
7,23056,RJ,30
6,23056,AC,1
9,72915,GO,9
8,72915,DF,1


In [21]:
geolocation_postcode = (
    geolocation_clean
    .groupby("geolocation_zip_code_prefix")
    .agg(
         geolocation_lat=("geolocation_lat", "median"),
        geolocation_lng=("geolocation_lng", "median"),
        geolocation_city=("geolocation_city", lambda x: x.mode().iloc[0]),
        geolocation_state=("geolocation_state", lambda x: x.mode().iloc[0])
    )
    .reset_index()
)

In [22]:
print("Rows",geolocation_postcode.shape[0])
print(
    "Duplicate post codes",
    geolocation_postcode["geolocation_zip_code_prefix"].duplicated().sum()
)

Rows 19015
Duplicate post codes 0


19,015 unique postcode after aggregation

In [23]:
geolocation_postcode["geolocation_zip_code_prefix"].duplicated().sum()

0

In [24]:
products_clean = products.copy()

In [25]:
products_clean = products_clean.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
    "product_length_cm":"product_length_cm"
})

In [26]:
products_clean.columns

Index(['product_id', 'product_category_name', 'product_name_length',
       'product_description_length', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='object')

in data profiling i found 610 products with missing category information

In [27]:
products_clean["missing_product_metadata"] = (
    products_clean["product_category_name"].isna()
)

products_clean["product_category_name"] = (
    products_clean["product_category_name"]
    .fillna("unknown")
)

In [28]:
print("Products flagged:", products_clean["missing_product_metadata"].sum())
print("Missing categories remaining:", products_clean["product_category_name"].isna().sum())

Products flagged: 610
Missing categories remaining: 0


in data profiling i found 4 products have weight = 0, and 2 products have missing physical weight 

In [32]:
products_clean["zero_product_weight"] = (
    products_clean["product_weight_g"] == 0
)
products_clean["missing_product_dimensions"] = (
    products_clean[
    [
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
  ]
    .isna()
    .any(axis = 1)
)   
print("zero_weight_products",products_clean["zero_product_weight"].sum())
print("products with missing dimensions:",products_clean["missing_product_dimensions"].sum())

zero_weight_products 4
products with missing dimensions: 2


In [33]:
category_translation_clean = category_translation.copy()
translated_categories = set(
    category_translation_clean["product_category_name"]
)
products_clean["missing_english_translation"]=(
    (products_clean["product_category_name"] != "unknown") &
    (~products_clean["product_category_name"].isin(translated_categories))
)

In [35]:
products_clean.loc[
    products_clean["missing_english_translation"],
    "product_category_name"
].value_counts()

product_category_name
portateis_cozinha_e_preparadores_de_alimentos    10
pc_gamer                                          3
Name: count, dtype: int64

In [36]:
orders_clean = orders.copy()

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
orders_clean[date_columns] = orders_clean[date_columns].apply(pd.to_datetime)

In [39]:
orders_clean[date_columns].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

In [40]:
orders_clean["delivered_missing_delivery_date"] = (
    (orders_clean["order_status"] == "delivered") &
    (orders_clean["order_delivered_customer_date"].isna())
)
orders_clean["non_delivered_with_delivery_date"] = (
    (orders_clean["order_status"] != "delivered") &
    (orders_clean["order_delivered_customer_date"].notna())
)
orders_clean["delivery_before_carrier"] = (
    orders_clean["order_delivered_customer_date"].notna()&
    orders_clean["order_delivered_carrier_date"].notna()&
     (
         orders_clean["order_delivered_customer_date"] <
         orders_clean["order_delivered_carrier_date"]
     )
)
orders_clean["late_delivery"] = (
    (orders_clean["order_status"] == "delivered")&
    orders_clean["order_delivered_customer_date"].notna()&
    (
        orders_clean["order_delivered_customer_date"].dt.normalize() >
        orders_clean["order_estimated_delivery_date"].dt.normalize()
    )
)

In [44]:
print(
    "delivered missing delivery date",
    orders_clean["delivered_missing_delivery_date"].sum()
)
print(
    "non delivered with delivery date",
    orders_clean["non_delivered_with_delivery_date"].sum()
)
print(
    "delivery before carrier",
    orders_clean["delivery_before_carrier"].sum()
)
print(
    "late deliveries",
    orders_clean["late_delivery"].sum()
)

delivered missing delivery date 8
non delivered with delivery date 6
delivery before carrier 23
late deliveries 6534


In [52]:
orders_clean["days_late"] = pd.NA

valid_delivery = (
    (orders_clean["order_status"] == "delivered") &
    orders_clean["order_delivered_customer_date"].notna()
)
orders_clean.loc[valid_delivery,"days_late"] = (
    orders_clean.loc[
        valid_delivery,
        "order_delivered_customer_date"
    ].dt.normalize()
    -
    orders_clean.loc[
        valid_delivery,
        "order_estimated_delivery_date"
    ].dt.normalize()
).dt.days

orders_clean.loc[
    valid_delivery & (orders_clean["days_late"] < 0),
    "days_late"
] = 0

In [53]:
orders_clean["days_late"].describe()

count     96470
unique      116
top           0
freq      89936
Name: days_late, dtype: int64

most orders were not late but some had significant delays. the maximum delay was 188 days.

In [54]:
orders_clean["days_late"] = (
    pd.to_numeric(
        orders_clean["days_late"],
        errors = "coerce"
    )
    .astype("Int64")
)

print(orders_clean["days_late"].dtype)
orders_clean["days_late"].describe()

Int64


count     96470.0
mean     0.719312
std      4.652564
min           0.0
25%           0.0
50%           0.0
75%           0.0
max         188.0
Name: days_late, dtype: Float64

In [55]:
payments_clean = payments.copy()

In [59]:
payments_clean["zero_payment_value"] = (
    payments_clean["payment_value"] == 0
)

payments_clean["zero_installments"] = (
    payments_clean["payment_installments"] == 0
)

payments_clean["undefined_payment_type"] = (
    payments_clean["payment_type"] == "not_defined"
)

In [60]:
print("Zero payment value:", payments_clean["zero_payment_value"].sum())
print("Zero installments:", payments_clean["zero_installments"].sum())
print("Undefined payment type:", payments_clean["undefined_payment_type"].sum())

Zero payment value: 9
Zero installments: 2
Undefined payment type: 3


in profiling i found 80 orders where the first available payment_sequential value starts at 2 instead of 1

In [62]:
first_payment_sequence = (
    payments_clean
    .groupby("order_id")["payment_sequential"]
    .min()
)
sequence_issue_orders = first_payment_sequence[
    first_payment_sequence > 1
].index
payments_clean["payment_sequence_issue"] = (
    payments_clean["order_id"].isin(sequence_issue_orders)
)
# for validate
print(
    "orders with payment sequence issue",
    payments_clean.loc[
        payments_clean["payment_sequence_issue"],
        "order_id"
    ].nunique()
)

orders with payment sequence issue 80


in profiling i found that 1 order exists in orders but has no matching record in payments

In [64]:
orders_clean["missing_payment_record"] = (
    ~orders_clean["order_id"].isin(payments_clean["order_id"])
)

In [65]:
print(
    "orders with no payment record",
    orders_clean["missing_payment_record"].sum()
)

orders with no payment record 1


In [66]:
reviews_clean = reviews.copy()

In [67]:
reviews_clean["review_creation_date"] = pd.to_datetime(
    reviews_clean["review_creation_date"]
)
reviews_clean["review_answer_timestamp"] = pd.to_datetime(
    reviews_clean["review_answer_timestamp"]
)

In [69]:
reviews_clean[
    ["review_creation_date","review_answer_timestamp"]
].dtypes

review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object

In [70]:
review_count_per_order = (
    reviews_clean
    .groupby("order_id")["review_id"]
    .transform("count")
)
reviews_clean["multiple_reviews_for_order"] = (
    review_count_per_order > 1
)

In [71]:
# for validating
print(
    "orders with multiple reviews",
    reviews_clean.loc[
        reviews_clean["multiple_reviews_for_order"],
        "order_id"
    ].nunique()
)

orders with multiple reviews 547


In [72]:
reviews_clean["repeated_review_id"] = (
    reviews_clean["review_id"].duplicated(keep = False)
)

In [73]:
print(
    "repeated review IDs",
    reviews_clean.loc[
        reviews_clean["repeated_review_id"],
        "review_id"
    ].nunique()
)
print(
    "rows involving repeated review IDs",
    reviews_clean["repeated_review_id"].sum()
)
    

repeated review IDs 789
rows involving repeated review IDs 1603


In [74]:
order_items_clean = order_items.copy()

In [77]:
order_items_clean["shipping_limit_date"] = pd.to_datetime(
    order_items_clean["shipping_limit_date"]
)

In [79]:
order_items_clean["shipping_limit_date"].dtype

dtype('<M8[ns]')

in profiling i found that 775 order exist in the orders table but no matching row in order_items

In [83]:
orders_clean["missing_order_items"] = (
    ~orders_clean["order_id"].isin(order_items_clean["order_id"])
)

In [84]:
print(
    "orders with no item records",
    orders_clean["missing_order_items"].sum()
)

orders with no item records 775


In [85]:
customers_clean["missing_geolocation"] = (
    ~customers_clean["customer_zip_code_prefix"]
    .isin(geolocation_postcode["geolocation_zip_code_prefix"])
)

In [87]:
print(
    "customers with no matching geolocation",
    customers_clean["missing_geolocation"].sum()
)

customers with no matching geolocation 278


In [88]:
sellers_clean["missing_geolocation"] = (
    ~sellers_clean["seller_zip_code_prefix"]
    .isin(geolocation_postcode["geolocation_zip_code_prefix"])
)

In [89]:
print(
    "sellers with no matching geolocation",
    sellers_clean["missing_geolocation"].sum()
)

sellers with no matching geolocation 7


In [90]:
print(
    "Duplicate categories names",
    category_translation_clean["product_category_name"].duplicated().sum()
)

Duplicate categories names 0


In [91]:
products_clean = products_clean.merge(
    category_translation_clean,
    on = "product_category_name",
    how = "left"
)

In [92]:
print("product rows",products_clean.shape[0])
print(
    "missing english category",
    products_clean["product_category_name_english"].isna().sum()
)

product rows 32951
missing english category 623


In [93]:
print("Customers rows:", customers_clean.shape[0])
print("Orders rows:", orders_clean.shape[0])
print("Order Items rows:", order_items_clean.shape[0])
print("Products rows:", products_clean.shape[0])
print("Payments rows:", payments_clean.shape[0])
print("Reviews rows:", reviews_clean.shape[0])
print("Sellers rows:", sellers_clean.shape[0])
print("Geolocation postcode rows:", geolocation_postcode.shape[0])

print() 

print("Duplicate primary keys:")
print("Customer ID duplicates:", customers_clean["customer_id"].duplicated().sum())
print("Order ID duplicates:", orders_clean["order_id"].duplicated().sum())
print("Product ID duplicates:", products_clean["product_id"].duplicated().sum())
print("Seller ID duplicates:", sellers_clean["seller_id"].duplicated().sum())
print(
    "Geolocation postcode duplicates:",
    geolocation_postcode["geolocation_zip_code_prefix"].duplicated().sum()
)

Customers rows: 99441
Orders rows: 99441
Order Items rows: 112650
Products rows: 32951
Payments rows: 103886
Reviews rows: 99224
Sellers rows: 3095
Geolocation postcode rows: 19015

Duplicate primary keys:
Customer ID duplicates: 0
Order ID duplicates: 0
Product ID duplicates: 0
Seller ID duplicates: 0
Geolocation postcode duplicates: 0


In [95]:
cleaning_log = pd.DataFrame({
    "table":[
        "Customers",
        "Sellers",
        "Geolocation",
        "Geolocation",
        "Products",
        "Products",
        "Products",
        "Orders",
        "Payments",
        "Reviews",
        "Order Items"
    ],
    "issue":[
        "postcode stored as a number",
        "postcode stored as a number",
        "exact duplicate rows",
        "multiple rows per postcode",
        "missing product category",
        "zero or missing weight/dimensions",
        "missing english category translation",
        "order date anomalies",
        "payment anomalies",
        "multiple repeated reviews",
        "shipping date stored as text"
    ],
    "action":[
        "converted postcode to 5 character string",
        "converted postcode to 5 character string",
        "remove extra duplicated rows",
        "aggregated to one row per postcode",
        "replaced missing category with unknown and created flag",
        "kept values and created quality flags",
        "created missing translation flag and added english category",
        "created quality flags and days_late column",
        "created anomaly flags",
        "created review quality flags",
        "converted shipping_limit_date to date_time"
    ],
    "reason":[
        "preserve leading zeros and support correct joins",
        "preserve leading zeros and support correct joins",
        "exact duplicates add no useful information",
        "prevent row multiplication during joins",
        "keep products while identifying missing metadata",
        "do not guess or delete uncertain values",
        "keep untranslated product visible",
        "keep unusual orders without deleting them",
        "keep unusual payments without inventing values",
        "avoid deleting geninue multiple reviews",
        "allow correct date calculation"
    ]
})

cleaning_log

,table,issue,action,reason
0,Customers,postcode stored as a number,converted postcode to 5 character string,preserve leading zeros and support correct joins
1,Sellers,postcode stored as a number,converted postcode to 5 character string,preserve leading zeros and support correct joins
2,Geolocation,exact duplicate rows,remove extra duplicated rows,exact duplicates and no useful information
3,Geolocation,multiple rows per postcode,aggregated to one row per postcode,prevent row multiplication during joins
4,Products,missing product category,replaced missing category with unknown and cre...,keep products while identifying missing metadata
5,Products,zero or missing weight/dimensions,kept values and created quality flags,do not guess or delete uncertain values
6,Products,missing english category translation,created missing translation flag and added eng...,keep untranslated product visible
7,Orders,order date anomalies,created quality flags and days_late column,keep unusual orders without deleting them
8,Payments,payment anomalies,created anomaly flags,keep unusual payments without inventing values
9,Reviews,multiple repeated reviews,created review quality flags,avoid deleting geninue multiple reviews


In [97]:
import os

os.makedirs("../06_clean_data", exist_ok=True)

### Saved all the clean tables

In [98]:
customers_clean.to_csv("../06_clean_data/customers_clean.csv", index=False)
orders_clean.to_csv("../06_clean_data/orders_clean.csv", index=False)
order_items_clean.to_csv("../06_clean_data/order_items_clean.csv", index=False)
products_clean.to_csv("../06_clean_data/products_clean.csv", index=False)
payments_clean.to_csv("../06_clean_data/payments_clean.csv", index=False)
reviews_clean.to_csv("../06_clean_data/reviews_clean.csv", index=False)
sellers_clean.to_csv("../06_clean_data/sellers_clean.csv", index=False)
geolocation_postcode.to_csv("../06_clean_data/geolocation_clean.csv", index=False)
category_translation_clean.to_csv("../06_clean_data/category_translation_clean.csv", index=False)
cleaning_log.to_csv("../06_clean_data/cleaning_log.csv", index=False)

print("all cleaned files saved successfully.")

all cleaned files saved successfully.


In [99]:
import os

os.listdir("../06_clean_data")

['category_translation_clean.csv',
 'cleaning_log.csv',
 'customers_clean.csv',
 'geolocation_clean.csv',
 'orders_clean.csv',
 'order_items_clean.csv',
 'payments_clean.csv',
 'products_clean.csv',
 'reviews_clean.csv',
 'sellers_clean.csv']